In [2]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import pickle

df = pd.read_csv('transaksi.csv')

df = df.dropna(subset=['CustomerID', 'Description'])
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
df = df[(df['Quantity'] > 0) & (df['Price'] > 0)]

df['YearMonth'] = df['InvoiceDate'].dt.to_period('M')

product_sales = df.groupby(['Description', 'YearMonth'])['Quantity'].sum().reset_index()
product_sales['YearMonth'] = product_sales['YearMonth'].astype(str)

produk_pilih = "WHITE HANGING HEART T-LIGHT HOLDER"

data = product_sales[
    product_sales['Description'].str.contains(produk_pilih, case=False, na=False)
].copy()

data = data.sort_values('YearMonth').reset_index(drop=True)

data['t'] = np.arange(len(data))
data['month'] = pd.to_datetime(data['YearMonth']).dt.month
data['year'] = pd.to_datetime(data['YearMonth']).dt.year

max_lag = min(3, len(data) - 1)   # karena data cuma 5 bulan

for i in range(1, max_lag + 1):
    data[f'lag{i}'] = data['Quantity'].shift(i)

data = data.dropna().reset_index(drop=True)

fitur = ['t', 'month', 'year'] + [f'lag{i}' for i in range(1, max_lag + 1)]

X = data[fitur]
y = data['Quantity']

model = RandomForestRegressor(
    n_estimators=200,
    max_depth=10,
    random_state=42
)

model.fit(X, y)

y_pred = model.predict(X)

print("MAE:", mean_absolute_error(y, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y, y_pred)))
print("R2:", r2_score(y, y_pred))

pickle.dump({
    'model': model,
    'fitur': fitur,
    'max_lag': max_lag
}, open('model.pkl', 'wb'))

print("Model tersimpan")
future = []

last = data.iloc[-1].copy()

for i in range(1):

    next_t = last['t'] + 1
    next_date = pd.to_datetime(last['YearMonth']) + pd.DateOffset(months=1)

    new_row = {
        't': next_t,
        'month': next_date.month,
        'year': next_date.year
    }

    for i in range(1, max_lag + 1):
        if i == 1:
            new_row[f'lag{i}'] = last['Quantity']
        else:
            new_row[f'lag{i}'] = last[f'lag{i-1}']

    new_df = pd.DataFrame([new_row])

    pred = model.predict(new_df)[0]

    future.append({
        "month": next_date.strftime('%Y-%m'),
        "prediction": float(pred)
    })

    print(future)

MAE: 207.30249999999978
RMSE: 208.12390483195318
R2: 0.7429500000000006
Model tersimpan
[{'month': '2026-06', 'prediction': 4900.18}]
